In [1]:
import jax
jax.config.update("jax_enable_x64", True)
from fewjaxstudy.trajectory.ode import get_trajectory_generator
import jax.numpy as jnp

from few.utils.constants import YRSID_SI, MTSUN_SI

In [3]:
generator = get_trajectory_generator("../amps-prep/KerrEccEqFluxData.h5")

In [5]:
m1 = 1e5
m2 = 1e1
a = 0.7
p0 = 10.0
e0 = 0.5
T = 1.0

M = m1 + m2
Msec = M * MTSUN_SI
mu = m1 * m2 / M
nu = mu / M
T_in = T * YRSID_SI / Msec * nu

sol = generator(m1, m2, a, p0, e0, 0., 0., T_in)

In [7]:
sol.stats["num_accepted_steps"], sol.interpolation.ts

(Array(69, dtype=int64, weak_type=True),
 Array([0.00000000e+00, 1.00000000e-02, 9.52160674e-02, 7.47107749e-01,
        4.13550877e+00, 1.89606683e+01, 4.33409985e+01, 7.06988010e+01,
        9.23317300e+01, 1.10982307e+02, 1.29632883e+02, 1.42873251e+02,
        1.54178647e+02, 1.63000656e+02, 1.72938972e+02, 1.79333938e+02,
        1.84535691e+02, 1.89172227e+02, 1.93308827e+02, 1.96444293e+02,
        1.99579759e+02, 2.01964648e+02, 2.04093716e+02, 2.05308164e+02,
        2.06443278e+02, 2.07578391e+02, 2.08713505e+02, 2.09448537e+02,
        2.09899605e+02, 2.10350674e+02, 2.10725444e+02, 2.10997636e+02,
        2.11224037e+02, 2.11477735e+02, 2.11614890e+02, 2.11777179e+02,
        2.11881562e+02, 2.11972903e+02, 2.12029822e+02, 2.12086741e+02,
        2.12125245e+02, 2.12160880e+02, 2.12195782e+02, 2.12219218e+02,
        2.12237535e+02, 2.12252860e+02, 2.12265924e+02, 2.12277547e+02,
        2.12286705e+02, 2.12296495e+02, 2.12302360e+02, 2.12307709e+02,
        2.12312347e+02,

In [11]:
solve_dynamics_jit = jax.jit(generator)
solve_many_dynamics = jax.vmap(solve_dynamics_jit, in_axes=(None, None, None, 0, None, None, None, None), out_axes=0)
_ = solve_dynamics_jit(m1, m2, a, p0, e0, 0., 0., T_in)

In [13]:
ps = jnp.linspace(10.0, 11.0, 10)
many = solve_many_dynamics(m1, m2, a, ps, e0, 0.0, 0.0, T)

In [14]:
jax.vmap(lambda s: s.evaluate(10.09))(many)

Array([[ 9.8947055 ,  0.49231444,  0.22258141,  0.16597945],
       [10.00948145,  0.49266497,  0.21863697,  0.16365409],
       [10.12408331,  0.49299539,  0.21481281,  0.16138243],
       [10.2385217 ,  0.4933071 ,  0.21110356,  0.15916273],
       [10.35280647,  0.49360137,  0.20750416,  0.15699332],
       [10.4669467 ,  0.49387938,  0.20400986,  0.1548726 ],
       [10.58095082,  0.49414222,  0.20061619,  0.15279903],
       [10.69482668,  0.4943909 ,  0.19731893,  0.15077114],
       [10.80858153,  0.49462633,  0.1941141 ,  0.14878753],
       [10.92222214,  0.49484936,  0.19099794,  0.14684684]],      dtype=float64)